# MedGemma Feasibility Check

This notebook is to verify the feasibility of using MedGemma model family/lineage in evaluation and interpretability methodology.

The purpose of this notebook is to validate:
* Access to opensource model e.g HuggingFace/Transformers
* Check hidden states
* Probe feasibility
* Test Pipeline and API use and format

# Attempt to import MedGemma Model: Medgemma-27b-it

Documentation/References for implementation:
1. https://huggingface.co/google/medgemma-27b-text-it
2. https://colab.research.google.com/github/google-health/medgemma/blob/main/notebooks/quick_start_with_hugging_face.ipynb

In [1]:
import os
import sys

google_colab = "google.colab" in sys.modules and not os.environ.get("VERTEX_PRODUCT")

if google_colab:
    # Use secret if running in Google Colab
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
else:
    # Store Hugging Face data under `/content` if running in Colab Enterprise
    if os.environ.get("VERTEX_PRODUCT") == "COLAB_ENTERPRISE":
        os.environ["HF_HOME"] = "/content/hf"
    # Authenticate with Hugging Face
    from huggingface_hub import get_token
    if get_token() is None:
        from huggingface_hub import notebook_login
        notebook_login()

In [2]:
! pip install --upgrade --quiet accelerate bitsandbytes transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 89.9 MB/s eta 0:00:00


In [3]:
from transformers import BitsAndBytesConfig
import torch

model_variant = "medgemma-1.5-4b-it"  # @param ["medgemma-1.5-4b-it", "medgemma-4b-it", "medgemma-27b-it", "medgemma-27b-text-it"]
model_id = f"google/{model_variant}"

# @markdown ---
# @markdown To reduce memory usage, [quantize](https://huggingface.co/docs/transformers/en/quantization/bitsandbytes) the model in 4-bit. **Note:** 4-bit quantization is required to run MedGemma 27B on an A100 GPU in Google Colab. The performance of quantized versions has not been evaluated.
use_quantization = True  # @param {type: "boolean"}

# @markdown ---
# @markdown MedGemma 1.5 4B and MedGemma 27B variants support thinking. If enabled, a system instruction will be added to prompt the model to output its reasoning between two special tokens `<unused94>` and `<unused95>` followed by the final response. **Note:** This notebook shows how to enable thinking for any multimodal or text-only task but it may not benefit some use cases. Thinking may potentially improve results on challenging text-only tasks at the cost of generating more tokens. Thoroughly validate the model's performance and assess the inference time and cost when enabling thinking.
is_thinking = False  # @param {type: "boolean"}
THINKING_VARIANTS = [
    "medgemma-1.5-4b-it",
    "medgemma-27b-it",
    "medgemma-27b-text-it",
]
if is_thinking and model_variant not in THINKING_VARIANTS:
    is_thinking = False
    print(
        "Note: Thinking is enabled for a non-thinking variant. Setting "
        "`is_thinking` to `False`."
    )

# If running a 27B variant in Google Colab, check if the runtime satisfies
# memory requirements
if "27b" in model_variant and google_colab:
    if not ("A100" in torch.cuda.get_device_name(0) and use_quantization):
        raise ValueError(
            "Runtime has insufficient memory to run a 27B variant. "
            "Please select an A100 GPU and use 4-bit quantization."
        )

model_kwargs = dict(
    dtype=torch.bfloat16,
    device_map="auto",
)

if use_quantization:
    model_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True)

## Loading of Pipeline

In [4]:
from transformers import pipeline

if "text" in model_variant:
    pipe = pipeline("text-generation", model=model_id, model_kwargs=model_kwargs)
else:
    pipe = pipeline("image-text-to-text", model=model_id, model_kwargs=model_kwargs)

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

## Load Model Directly

In [5]:
if "text" in model_variant:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
else:
    from transformers import AutoModelForImageTextToText, AutoProcessor
    model = AutoModelForImageTextToText.from_pretrained(model_id, **model_kwargs)
    processor = AutoProcessor.from_pretrained(model_id) # AutoProcessor

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

### If running text-to-text model, to raise ValueError

In [ ]:
if "text" in model_variant:
    raise ValueError(
        "You are using a text-only variant which does not support multimodal "
        "inputs. Please proceed to the 'Run inference on text only' section."
    )

# Feasibility Check 1: Test Medical Image and Supporting Text

### Attempt 1: Had issues loading web image to run directly

In [6]:
import os
from PIL import Image
from IPython.display import Image as IPImage, display, Markdown

prompt = "Describe this X-ray"  # @param {type: "string"}

# Image attribution: Another publicly available chest X-ray from Wikimedia Commons (direct link)
image_url = "https://upload.wikimedia.org/wikipedia/commons/c/cd/Chest_Xray_Medical_Imaging_Radiology_2004.jpg"  # @param {type: "string"}

!wget -nc {image_url}
image_filename = os.path.basename(image_url)

if os.path.exists(image_filename):
    image = Image.open(image_filename)
    print(f"Image '{image_filename}' loaded successfully.")
else:
    print(f"Error: Image '{image_filename}' not found. Please check the URL and your network connection.")
    image = None # Set image to None to avoid further errors if not found

--2026-05-26 07:40:24--  https://upload.wikimedia.org/wikipedia/commons/c/cd/Chest_Xray_Medical_Imaging_Radiology_2004.jpg
Resolving upload.wikimedia.org (upload.wikimedia.org)... 208.80.154.240, 2620:0:861:ed1a::2:b
Connecting to upload.wikimedia.org (upload.wikimedia.org)|208.80.154.240|:443... connected.
HTTP request sent, awaiting response... 429 Too Many Requests (015d12c)
2026-05-26 07:40:24 ERROR 429: Too Many Requests (015d12c).

Error: Image 'Chest_Xray_Medical_Imaging_Radiology_2004.jpg' not found. Please check the URL and your network connection.


### Attemp 2: To upload a chest X-ray image directly to drive to test

Link: https://upload.wikimedia.org/wikipedia/commons/c/c8/Chest_Xray_PA_3-8-2010.png

In [7]:
import os
from PIL import Image
from IPython.display import Image as IPImage, display, Markdown

prompt = "Describe this X-ray"  # @param {type: "string"}

# Replace 'your_uploaded_image.jpg' with the actual filename of your uploaded image
image_filename = "/content/drive/MyDrive/TARA Capstone/Chest_Xray_PA_3-8-2010.png" # @param {type: "string"}

if os.path.exists(image_filename):
    image = Image.open(image_filename)
    print(f"Image '{image_filename}' loaded successfully.")
    display(IPImage(filename=image_filename))
else:
    print(f"Error: Image '{image_filename}' not found. Please make sure you have uploaded the file and entered the correct filename.")
    image = None # Set image to None to avoid further errors if not found

Output hidden; open in https://colab.research.google.com to view.

In [8]:
role_instruction = "You are an expert radiologist."
if is_thinking:
    system_instruction = f"SYSTEM INSTRUCTION: think silently if needed. {role_instruction}"
    max_new_tokens = 1300
else:
    system_instruction = role_instruction
    max_new_tokens = 300

messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": system_instruction}]
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": prompt},
            {"type": "image", "image": image}
        ]
    }
]

In [9]:
output = pipe(text=messages, max_new_tokens=max_new_tokens, do_sample=False)
response = output[0]["generated_text"][-1]["content"]

display(Markdown(f"---\n\n**[ User ]**\n\n{prompt}"))
display(IPImage(filename=image_filename, height=300))
if is_thinking:
    thought, response = response.split("<unused95>")
    thought = thought.replace("<unused94>thought\n", "")
    display(Markdown(f"---\n\n**[ MedGemma thinking trace ]**\n\n{thought}"))
display(Markdown(f"---\n\n**[ MedGemma ]**\n\n{response}\n\n---"))

Output hidden; open in https://colab.research.google.com to view.

### Run Model Direct

In [10]:
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device, dtype=torch.bfloat16)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generation = generation[0][input_len:]

response = processor.decode(generation, skip_special_tokens=True)

display(Markdown(f"---\n\n**[ User ]**\n\n{prompt}"))
display(IPImage(filename=image_filename, height=300))
if is_thinking:
    thought, response = response.split("<unused95>")
    thought = thought.replace("<unused94>thought\n", "")
    display(Markdown(f"---\n\n**[ MedGemma thinking trace ]**\n\n{thought}"))
display(Markdown(f"---\n\n**[ MedGemma ]**\n\n{response}\n\n---"))

Output hidden; open in https://colab.research.google.com to view.

### Observations/Inishgts/Actionables

1. Took about 5 mins to load MEdGemma Model

# Feasibility Check 2: Test Health Text to Text Generation

In [11]:
from IPython.display import Markdown

prompt = "How do you differentiate bacterial from viral pneumonia?"  # @param {type: "string"}

role_instruction = "You are a helpful medical assistant."
if is_thinking:
    system_instruction = f"SYSTEM INSTRUCTION: think silently if needed. {role_instruction}"
    max_new_tokens = 1500
else:
    system_instruction = role_instruction
    max_new_tokens = 500

messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": system_instruction}]
    },
    {
        "role": "user",
        "content": [{"type": "text", "text": prompt}]
    }
]

In [12]:
output = pipe(messages, max_new_tokens=max_new_tokens, do_sample=False)
response = output[0]["generated_text"][-1]["content"]

display(Markdown(f"---\n\n**[ User ]**\n\n{prompt}\n\n---"))
if is_thinking:
    thought, response = response.split("<unused95>")
    thought = thought.replace("<unused94>thought\n", "")
    display(Markdown(f"**[ MedGemma thinking trace ]**\n\n{thought}\n\n---"))
display(Markdown(f"**[ MedGemma ]**\n\n{response}\n\n---"))

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


---

**[ User ]**

How do you differentiate bacterial from viral pneumonia?

---

**[ MedGemma ]**

As a helpful medical assistant, I can provide information on how to differentiate between bacterial and viral pneumonia, but it's crucial to understand that **I cannot provide medical diagnoses.** This information is for educational purposes only and should not replace professional medical advice from a qualified healthcare provider.

Here's a general overview of how doctors differentiate between bacterial and viral pneumonia, keeping in mind that these are often overlapping conditions and a definitive diagnosis usually requires specific tests:

**Key Differences and Diagnostic Approaches:**

1.  **Clinical Presentation (Symptoms):**
    *   **Both:** Can present with fever, cough (often productive with thick sputum), shortness of breath, chest pain, fatigue, chills, and sometimes confusion (especially in the elderly).
    *   **Bacterial:** Often associated with a more abrupt onset and severe symptoms. Sputum is typically purulent (thick, yellow/green).
    *   **Viral:** Onset can be more gradual. Sputum may be less thick or even dry.

2.  **Risk Factors:**
    *   **Bacterial:** Often associated with preceding upper respiratory infections (like the flu, common cold), exposure to certain bacteria (like *Streptococcus pneumoniae*, *Haemophilus influenzae*, *Mycoplasma pneumoniae*, *Legionella pneumophila*), underlying lung conditions (like COPD, cystic fibrosis), immunosuppression, or recent hospitalization.
    *   **Viral:** Often associated with influenza, RSV, COVID-19, adenovirus, metapneumovirus, etc. Can occur in anyone, but certain groups (young children, elderly, immunocompromised) are at higher risk.

3.  **Physical Examination Findings:**
    *   **Both:** May show signs of respiratory distress (tachypnea, tachycardia, hypoxia), crackles or rales on auscultation (listening to the lungs), and potentially signs of consolidation (like dullness to percussion or increased tactile fremitus).
    *   **Bacterial:** May show signs of consolidation (e.g., lobar pneumonia).
    *   **Viral:** May show diffuse interstitial patterns (e.g., atypical pneumonia).

4.  **Diagnostic Tests:**
    *   **Chest X-ray:**
        *   **Bacterial:** Often shows lobar consolidation (a dense area of opacity in one or more lobes), but can also be patchy or diffuse.
        *   **Viral:**

---

# Feasibility Check 3: Direct API Call for MedGemma

In [13]:
processor_or_tokenizer = tokenizer if "text" in model_variant else processor

inputs = processor_or_tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generation = generation[0][input_len:]

response = processor_or_tokenizer.decode(generation, skip_special_tokens=True)

display(Markdown(f"---\n\n**[ User ]**\n\n{prompt}\n\n---"))
if is_thinking:
    thought, response = response.split("<unused95>")
    thought = thought.replace("<unused94>thought\n", "")
    display(Markdown(f"**[ MedGemma thinking trace ]**\n\n{thought}\n\n---"))
display(Markdown(f"**[ MedGemma ]**\n\n{response}\n\n---"))

[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


---

**[ User ]**

How do you differentiate bacterial from viral pneumonia?

---

**[ MedGemma ]**

As a helpful medical assistant, I can provide information on how to differentiate between bacterial and viral pneumonia, but it's crucial to understand that **I cannot provide medical diagnoses.** This information is for educational purposes only and should not replace professional medical advice from a qualified healthcare provider.

Here's a general overview of how doctors differentiate between bacterial and viral pneumonia, keeping in mind that these are often overlapping conditions and a definitive diagnosis usually requires specific tests:

**Key Differences and Diagnostic Approaches:**

1.  **Clinical Presentation (Symptoms):**
    *   **Both:** Can present with fever, cough (often productive with thick sputum), shortness of breath, chest pain, fatigue, chills, and sometimes confusion (especially in the elderly).
    *   **Bacterial:** Often associated with a more abrupt onset and severe symptoms. Sputum is typically purulent (thick, yellow/green).
    *   **Viral:** Onset can be more gradual. Sputum may be less thick or even dry.

2.  **Risk Factors:**
    *   **Bacterial:** Often associated with preceding upper respiratory infections (like the flu, common cold), exposure to certain bacteria (like *Streptococcus pneumoniae*, *Haemophilus influenzae*, *Mycoplasma pneumoniae*, *Legionella pneumophila*), underlying lung conditions (like COPD, cystic fibrosis), immunosuppression, or recent hospitalization.
    *   **Viral:** Often associated with influenza, RSV, COVID-19, adenovirus, metapneumovirus, etc. Can occur in anyone, but certain groups (young children, elderly, immunocompromised) are at higher risk.

3.  **Physical Examination Findings:**
    *   **Both:** May show signs of respiratory distress (tachypnea, tachycardia, hypoxia), crackles or rales on auscultation (listening to the lungs), and potentially signs of consolidation (like dullness to percussion or increased tactile fremitus).
    *   **Bacterial:** May show signs of consolidation (e.g., lobar pneumonia).
    *   **Viral:** May show diffuse interstitial patterns (e.g., atypical pneumonia).

4.  **Diagnostic Tests:**
    *   **Chest X-ray:**
        *   **Bacterial:** Often shows lobar consolidation (a dense area of opacity in one or more lobes), but can also be patchy or diffuse.
        *   **Viral:**

---

# Testing Specfic Components of MedGemma Family to Assess Suitability

* Access to Hidden States
* Ability to run probes
* (Maybe even can see how to run Mech Interp exercise)

 ### Component 1: Checking of Hidden State Access

In [ ]:
import os
import json
import math
import numpy as np
import pandas as pd
import torch

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

In [ ]:
prompt = "A patient has chest pain and shortness of breath. Should they seek urgent care?"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model(**inputs, output_hidden_states=True, use_cache=False)

hidden_states = out.hidden_states
last_hidden = hidden_states[-1]

print("Num hidden layers returned:", len(hidden_states))
print("Last hidden shape:", last_hidden.shape)

NameError: name 'tokenizer' is not defined